# SQL function wheel test

Install local `0.0.0` wheels of `cognite-pygen-spark` and `cognite-databricks`, register the cogsail `LimsResult` view, and create Unity Catalog SQL functions with `generator.create_sql_function`.

Upload both wheels to `/Workspace/Users/fredrik.holm@cognitedata.com/wheels/` before running the install cell. Build them from the checkouts with the pygen-spark dependency temporarily set to `>=0.0.0`, then restore the release pin. Do not commit that pin.

The `LimsResult` seed is already on cognite-cogsail: lab A has 60,000 nodes, lab B has 40,000, and each `componentName` is 6,000 rows in lab A and 10,000 rows across both labs.

`%pip` and `%restart_python` are their own cells. Run them, then continue from the version check.


In [ ]:
%pip install --force-reinstall /Workspace/Users/fredrik.holm@cognitedata.com/wheels/cognite_pygen_spark-0.0.0-py3-none-any.whl /Workspace/Users/fredrik.holm@cognitedata.com/wheels/cognite_databricks-0.0.0-py3-none-any.whl


In [ ]:
%restart_python


In [ ]:
import cognite.databricks as db
import cognite.pygen_spark as ps

print(ps.__version__, db.__version__)
assert ps.__version__ == "0.0.0"
assert db.__version__ == "0.0.0"


## Clients

Point `TOML_PATH` at the cogsail credentials file. A bearer token is enough to read CDF. Writing the secret scope still needs `client_id`, `client_secret`, and `tenant_id` in that file, or an existing scope from an earlier registration.


In [ ]:
import tomllib
from pathlib import Path

from cognite.client import CogniteClient
from cognite.client.config import ClientConfig
from cognite.client.credentials import OAuthClientCredentials, Token
from databricks.sdk import WorkspaceClient

TOML_PATH = "/Workspace/Users/fredrik.holm@cognitedata.com/toml/cog-sail.toml"

CATALOG = "f0connectortest"
SECRET_SCOPE = "cdf_dm_dom_lims_result_limsresult_dom"
DM_SPACE = "dm_dom_lims_result"
MODEL_EXTERNAL_ID = "LimsResult_DOM"
MODEL_VERSION = "v1"
SCHEMA = f"{DM_SPACE}_{MODEL_EXTERNAL_ID.lower()}_{MODEL_VERSION}"
VIEW = f"{CATALOG}.{SCHEMA}.LimsResult"
LAB_A = "inst_lims_result_lab_a"
LAB_B = "inst_lims_result_lab_b"
OUTPUT_DIR = "/Workspace/Users/fredrik.holm@cognitedata.com/udtf_generated"

with Path(TOML_PATH).open("rb") as fh:
    data = tomllib.load(fh)
cognite = data["cognite"]
runtime = data.get("fn_btp_hierarchy_runtime") or {}
triggers = data.get("hierarchy_workflow_triggers") or {}
secrets = data.get("hierarchy_workflow_trigger_secrets") or {}
client_id = runtime.get("client_id") or cognite.get("client_id") or triggers.get("ingest_trigger_client_id")
client_secret = (
    runtime.get("client_secret") or cognite.get("client_secret") or secrets.get("ingest_trigger_client_secret")
)
tenant_id = cognite.get("tenant_id") or cognite.get("idp_tenant_id")
base_url = f"https://{cognite['cdf_cluster']}.cognitedata.com"

if client_id and client_secret and tenant_id:
    credentials = OAuthClientCredentials(
        token_url=f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token",
        client_id=client_id,
        client_secret=client_secret,
        scopes=[f"{base_url}/.default"],
    )
else:
    bearer_token = cognite.get("bearer_token")
    if not isinstance(bearer_token, str) or not bearer_token:
        raise ValueError("TOML needs client credentials or cognite.bearer_token")
    credentials = Token(lambda: bearer_token)

client = CogniteClient(
    ClientConfig(
        client_name="sql-function-wheel-test",
        project=cognite["project"],
        credentials=credentials,
        base_url=base_url,
    )
)
workspace_client = WorkspaceClient()
print("project:", client.config.project)
print("base_url:", client.config.base_url)
print("view:", VIEW)


## Register the view

Replace the Python UDTF and the catalog view so the SQL function calls the wheel under test.


In [ ]:
from cognite.client.data_classes.data_modeling.ids import DataModelId
from cognite.databricks import generate_udtf_notebook

generator = generate_udtf_notebook(
    DataModelId(space=DM_SPACE, external_id=MODEL_EXTERNAL_ID, version=MODEL_VERSION),
    client,
    workspace_client=workspace_client,
    catalog=CATALOG,
    schema=SCHEMA,
    output_dir=OUTPUT_DIR,
)
if client_id and client_secret and tenant_id:
    generator.secret_helper.set_cdf_credentials(
        scope_name=SECRET_SCOPE,
        project=client.config.project,
        cdf_cluster=cognite["cdf_cluster"],
        client_id=client_id,
        client_secret=client_secret,
        tenant_id=tenant_id,
        base_url=client.config.base_url,
    )
else:
    print("Bearer token only. Reusing the secret scope from the last registration.")

secret_keys = sorted(item.key for item in workspace_client.secrets.list_secrets(scope=SECRET_SCOPE))
print("secret keys:", secret_keys)
assert "base_url" in secret_keys

udtfs = generator.register_udtfs(secret_scope=SECRET_SCOPE, if_exists="replace")
views = generator.register_views(secret_scope=SECRET_SCOPE, if_exists="replace")
print("registered UDTFs:", udtfs.total_count)
print("registered views:", [item.view_name for item in views.registered_views])


## Create SQL functions

Each call validates the statement with `rewrite_query`. Literals become arguments. Secrets stay `SECRET()` references. A statement that cannot be pushed raises `QueryNotPushdownCompatible` and creates nothing.


In [ ]:
from cognite.databricks.data_model_query_rewriter import QueryNotPushdownCompatible

count_by_space_and_component = generator.create_sql_function(
    "lims_result_count_by_space_and_component",
    f"""
    SELECT count(*) AS n
    FROM {VIEW}
    WHERE space = '{LAB_A}'
      AND componentName = 'MOISTURE_CONTENT'
    """,
    secret_scope=SECRET_SCOPE,
)
count_by_component = generator.create_sql_function(
    "lims_result_count_by_component",
    f"""
    SELECT componentName, count(*) AS n
    FROM {VIEW}
    WHERE space = '{LAB_A}'
    GROUP BY componentName
    """,
    secret_scope=SECRET_SCOPE,
)
count_in_spaces = generator.create_sql_function(
    "lims_result_count_by_component_in_spaces",
    f"""
    SELECT componentName, count(*) AS n
    FROM {VIEW}
    WHERE space IN ('{LAB_A}', '{LAB_B}')
    GROUP BY componentName
    """,
    secret_scope=SECRET_SCOPE,
)

for created in (count_by_space_and_component, count_by_component, count_in_spaces):
    print(created.full_name, [(arg.name, arg.sql_type) for arg in created.arguments])

try:
    generator.create_sql_function(
        "lims_result_bad_group",
        f"SELECT componentName, count(*) AS n FROM {VIEW} GROUP BY space",
        secret_scope=SECRET_SCOPE,
    )
    raise AssertionError("GROUP BY that does not match the select list must be rejected")
except QueryNotPushdownCompatible as exc:
    print("rejected:", exc)


## Call the functions

These counts are the cogsail seed. A slicer changes the argument. It does not query the view.


In [ ]:
moisture = spark.sql(
    f"""
    SELECT *
    FROM {count_by_space_and_component.full_name}(
      space => '{LAB_A}',
      componentName => 'MOISTURE_CONTENT'
    )
    """
).collect()
assert moisture[0]["count_externalId"] == "6000"
print("lab A moisture", moisture[0]["count_externalId"])

by_component = spark.sql(
    f"SELECT * FROM {count_by_component.full_name}(space => '{LAB_A}')"
).collect()
assert len(by_component) == 10
assert {row["count_externalId"] for row in by_component} == {"6000"}
assert {row["componentName"] for row in by_component} >= {"MOISTURE_CONTENT", "PH"}
print("lab A groups", len(by_component))

both_labs = spark.sql(
    f"""
    SELECT *
    FROM {count_in_spaces.full_name}(space => array('{LAB_A}', '{LAB_B}'))
    """
).collect()
assert len(both_labs) == 10
assert {row["count_externalId"] for row in both_labs} == {"10000"}
print("both labs groups", len(both_labs))

display(
    spark.sql(f"DESCRIBE FUNCTION EXTENDED {count_by_component.full_name}")
)
